1. Из ноутбуков по практике "Рекуррентные и одномерные сверточные нейронные сети" выберите лучшую сеть, либо создайте свою.
2. Запустите раздел "Подготовка"
3. Подготовьте датасет с параметрами `VOCAB_SIZE=20'000`, `WIN_SIZE=1000`, `WIN_HOP=100`, как в ноутбуке занятия, и обучите выбранную сеть. Параметры обучения можно взять из практического занятия. Для  всех обучаемых сетей в данной работе они должны быть одни и теже.
4. Поменяйте размер словаря tokenaizera (`VOCAB_SIZE`) на `5000`, `10000`, `40000`.  Пересоздайте датасеты, при этом оставьте `WIN_SIZE=1000`, `WIN_HOP=100`.
Обучите выбранную нейронку на этих датасетах.  Сделайте выводы об  изменении  точности распознавания авторов текстов. Результаты сведите в таблицу
5. Поменяйте длину отрезка текста и шаг окна разбиения текста на векторы  (`WIN_SIZE`, `WIN_HOP`) используя значения (`500`,`50`) и (`2000`,`200`). Пересоздайте датасеты, при этом оставьте `VOCAB_SIZE=20000`. Обучите выбранную нейронку на этих датасетах. Сделайте выводы об  изменении точности распознавания авторов текстов.

Результаты всей работы сведите в таблицу.

## Подготовка

## Импорт неоюходимых библиотек 

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
import re
import zipfile
import requests
import gdown
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [5]:
# Загрузим датасет из облака
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l7/writers.zip', None, quiet=True)

'writers.zip'

In [6]:
# Распакуем архив в папку writers
!unzip -o writers.zip -d writers/


Archive:  writers.zip
  inflating: writers/(Клиффорд_Саймак) Обучающая_5 вместе.txt  
  inflating: writers/(Клиффорд_Саймак) Тестовая_2 вместе.txt  
  inflating: writers/(Макс Фрай) Обучающая_5 вместе.txt  
  inflating: writers/(Макс Фрай) Тестовая_2 вместе.txt  
  inflating: writers/(О. Генри) Обучающая_50 вместе.txt  
  inflating: writers/(О. Генри) Тестовая_20 вместе.txt  
  inflating: writers/(Рэй Брэдберри) Обучающая_22 вместе.txt  
  inflating: writers/(Рэй Брэдберри) Тестовая_8 вместе.txt  
  inflating: writers/(Стругацкие) Обучающая_5 вместе.txt  
  inflating: writers/(Стругацкие) Тестовая_2 вместе.txt  
  inflating: writers/(Булгаков) Обучающая_5 вместе.txt  
  inflating: writers/(Булгаков) Тестовая_2 вместе.txt  


In [7]:
# Настройка констант для загрузки данных
FILE_DIR  = 'writers'                     # Папка с текстовыми файлами
SIG_TRAIN = 'обучающая'                   # Признак обучающей выборки в имени файла
SIG_TEST  = 'тестовая'                    # Признак тестовой выборки в имени файла

In [8]:
# Подготовим пустые списки

CLASS_LIST = []  # Список классов
text_train = []  # Список для оучающей выборки
text_test = []   # Список для тестовой выборки

# Получим списка файлов в папке
file_list = os.listdir(FILE_DIR)

for file_name in file_list:
    # Выделяем имя класса и типа выборки из имени файла
    m = re.match('\((.+)\) (\S+)_', file_name)
    # Если выделение получилось, то файл обрабатываем
    if m:

        # Получим имя класса
        class_name = m[1]

        # Получим имя выборки
        subset_name = m[2].lower()

        # Проверим тип выборки
        is_train = SIG_TRAIN in subset_name
        is_test = SIG_TEST in subset_name

        # Если тип выборки обучающая либо тестовая - файл обрабатываем
        if is_train or is_test:

            # Добавляем новый класс, если его еще нет в списке
            if class_name not in CLASS_LIST:
                print(f'Добавление класса "{class_name}"')
                CLASS_LIST.append(class_name)

                # Инициализируем соответствующих классу строки текста
                text_train.append('')
                text_test.append('')

            # Найдем индекс класса для добавления содержимого файла в выборку
            cls = CLASS_LIST.index(class_name)
            print(f'Добавление файла "{file_name}" в класс "{CLASS_LIST[cls]}", {subset_name} выборка.')

            # Откроем файл на чтение
            with open(f'{FILE_DIR}/{file_name}', 'r') as f:

                # Загрузим содержимого файла в строку
                text = f.read()
            # Определим выборку, куда будет добавлено содержимое
            subset = text_train if is_train else text_test

            # Добавим текста к соответствующей выборке класса. Концы строк заменяются на пробел
            subset[cls] += ' ' + text.replace('\n', ' ')

Добавление класса "Клиффорд_Саймак"
Добавление файла "(Клиффорд_Саймак) Тестовая_2 вместе.txt" в класс "Клиффорд_Саймак", тестовая выборка.
Добавление класса "Макс Фрай"
Добавление файла "(Макс Фрай) Обучающая_5 вместе.txt" в класс "Макс Фрай", обучающая выборка.
Добавление класса "Булгаков"
Добавление файла "(Булгаков) Тестовая_2 вместе.txt" в класс "Булгаков", тестовая выборка.
Добавление класса "Стругацкие"
Добавление файла "(Стругацкие) Тестовая_2 вместе.txt" в класс "Стругацкие", тестовая выборка.
Добавление файла "(Булгаков) Обучающая_5 вместе.txt" в класс "Булгаков", обучающая выборка.
Добавление файла "(Стругацкие) Обучающая_5 вместе.txt" в класс "Стругацкие", обучающая выборка.
Добавление файла "(Клиффорд_Саймак) Обучающая_5 вместе.txt" в класс "Клиффорд_Саймак", обучающая выборка.
Добавление класса "О. Генри"
Добавление файла "(О. Генри) Обучающая_50 вместе.txt" в класс "О. Генри", обучающая выборка.
Добавление файла "(Макс Фрай) Тестовая_2 вместе.txt" в класс "Макс Фрай", те

<>:12: SyntaxWarning: invalid escape sequence '\('
<>:12: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_4166/2214232407.py:12: SyntaxWarning: invalid escape sequence '\('
  m = re.match('\((.+)\) (\S+)_', file_name)


In [9]:
# Определим количество классов
CLASS_COUNT = len(CLASS_LIST)

In [10]:
# Выведем прочитанные классы текстов
print(CLASS_LIST)

['Клиффорд_Саймак', 'Макс Фрай', 'Булгаков', 'Стругацкие', 'О. Генри', 'Рэй Брэдберри']


In [11]:
# Посчитаем количество текстов в обучающей выборке
print(len(text_train))

6


In [12]:
# Проверим загрузки: выведем начальные отрывки из каждого класса

for cls in range(CLASS_COUNT):                   # Запустим цикл по числу классов
    print(f'Класс: {CLASS_LIST[cls]}')           # Выведем имя класса
    print(f'  train: {text_train[cls][:200]}')   # Выведем фрагмент обучающей выборки
    print(f'  test : {text_test[cls][:200]}')    # Выведем фрагмент тестовой выборки
    print()

Класс: Клиффорд_Саймак
  train:  ﻿Всё живое...     Когда я выехал из нашего городишка и повернул на шоссе, позади оказался грузовик. Этакая тяжелая громадина с прицепом, и неслась она во весь дух. Шоссе здесь срезает угол городка, и
  test :  ﻿Зачарованное паломничество    1  Гоблин со стропил следил за прячущимся монахом, который шпионил за ученым. Гоблин ненавидел монаха и имел для этого все основания. Монах никого не ненавидел и не люб

Класс: Макс Фрай
  train:  ﻿Власть несбывшегося   – С тех пор как меня угораздило побывать в этой грешной Черхавле, мне ежедневно снится какая-то дичь! – сердито сказал я Джуффину. – Сглазили они меня, что ли? А собственно, по
  test :  ﻿Слишком много кошмаров    Когда балансируешь над пропастью на узкой, скользкой от крови доске, ответ на закономерный вопрос: «Как меня сюда занесло?» – вряд ли принесёт практическую пользу. Зато пои

Класс: Булгаков
  train:  ﻿Белая гвардия   Посвящается[1]  Любови Евгеньевне Белозерской[2]  Пошел мелкий снег и вдруг

## Токенизация и создание словаря 

In [13]:
class TextTokenizer:
    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.word2idx = {"<PAD>": 0, "<UNK>": 1}
    
    def fit(self, texts):
        all_words = []
        for text in texts:
            words = re.findall(r'\w+', text.lower())
            all_words.extend(words)
        
        common_words = Counter(all_words).most_common(self.vocab_size - 2)
        for word, _ in common_words:
            self.word2idx[word] = len(self.word2idx)
            
    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            words = re.findall(r'\w+', text.lower())
            seq = [self.word2idx.get(w, 1) for w in words]
            sequences.append(seq)
        return sequences

## Подготовка Dataset и DataLoader

In [14]:
class WritersDataset(Dataset):
    def __init__(self, sequences, win_size, win_hop):
        self.x, self.y = [], []
        for cls_idx, seq in enumerate(sequences):
            for i in range(0, len(seq) - win_size + 1, win_hop):
                self.x.append(seq[i:i + win_size])
                self.y.append(cls_idx)
        self.x = torch.LongTensor(self.x)
        self.y = torch.LongTensor(self.y)

    def __len__(self): return len(self.x)
    def __getitem__(self, idx): return self.x[idx], self.y[idx]

## Архитектура модели

In [15]:
class WriterClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, class_count):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)

        self.lstm1 = nn.LSTM(emb_dim, 8, batch_first=True, bidirectional=True)
        self.lstm2 = nn.LSTM(16, 8, batch_first=True, bidirectional=True)
        self.gru1 = nn.GRU(16, 16, batch_first=True)
        self.gru2 = nn.GRU(16, 16, batch_first=True)
        
        self.classifier = nn.Sequential(
            nn.Linear(16, 200),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.BatchNorm1d(200),
            nn.Linear(200, class_count)
        )

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm1(x)
        x, _ = self.lstm2(x)
        x, _ = self.gru1(x)
        _, x = self.gru2(x) 
        return self.classifier(x.squeeze(0))

## Функция обучение и валидации 

In [16]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == y).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            correct += (outputs.argmax(1) == y).sum().item()
    return correct / len(loader.dataset)

In [18]:
VOCAB_SIZE = 20000
WIN_SIZE = 1000
WIN_HOP = 100
EPOCHS = 12


tokenizer = TextTokenizer(VOCAB_SIZE)
tokenizer.fit(text_train)
train_seq = tokenizer.texts_to_sequences(text_train)
test_seq = tokenizer.texts_to_sequences(text_test)

train_ds = WritersDataset(train_seq, WIN_SIZE, WIN_HOP)
test_ds = WritersDataset(test_seq, WIN_SIZE, WIN_HOP)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=128)


model = WriterClassifier(VOCAB_SIZE, 50, CLASS_COUNT).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


for epoch in range(EPOCHS):
    loss, acc = train_epoch(model, train_loader, criterion, optimizer)
    val_acc = evaluate(model, test_loader)
    print(f"Epoch {epoch+1}: Loss {loss:.4f}, Train Acc {acc:.4f}, Val Acc {val_acc:.4f}")

Epoch 1: Loss 1.6844, Train Acc 0.3113, Val Acc 0.3198
Epoch 2: Loss 1.3284, Train Acc 0.4657, Val Acc 0.3588
Epoch 3: Loss 0.7936, Train Acc 0.6705, Val Acc 0.3778
Epoch 4: Loss 0.4605, Train Acc 0.7947, Val Acc 0.4029
Epoch 5: Loss 0.3010, Train Acc 0.8553, Val Acc 0.3771
Epoch 6: Loss 0.1809, Train Acc 0.9376, Val Acc 0.4299
Epoch 7: Loss 0.0640, Train Acc 0.9883, Val Acc 0.4127
Epoch 8: Loss 0.0738, Train Acc 0.9787, Val Acc 0.4322
Epoch 9: Loss 0.0320, Train Acc 0.9965, Val Acc 0.4246
Epoch 10: Loss 0.0163, Train Acc 0.9986, Val Acc 0.4237
Epoch 11: Loss 0.0118, Train Acc 0.9989, Val Acc 0.4243
Epoch 12: Loss 0.0075, Train Acc 0.9997, Val Acc 0.4251


In [ ]:
import pandas as pd

def run_experiment(v_size, w_size, w_hop, epochs=20):
    print(f"\n--- Запуск: Vocab={v_size}, Win={w_size}, Hop={w_hop} ---")
    

    tokenizer = TextTokenizer(v_size)
    tokenizer.fit(text_train)
    train_seq = tokenizer.texts_to_sequences(text_train)
    test_seq = tokenizer.texts_to_sequences(text_test)

    train_ds = WritersDataset(train_seq, w_size, w_hop)
    test_ds = WritersDataset(test_seq, w_size, w_hop)
    

    batch_size = min(128, len(train_ds))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)


    model = WriterClassifier(v_size, 50, CLASS_COUNT).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    best_val_acc = 0
    

    for epoch in range(epochs):
        loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
        val_acc = evaluate(model, test_loader)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {loss:.4f} | Val Acc: {val_acc:.4f}")
            
    return best_val_acc

In [ ]:
vocab_results = []
VOCAB_LIST = [5000, 10000, 20000, 40000]
FIXED_WIN = 1000
FIXED_HOP = 100

for v_size in VOCAB_LIST:
    acc = run_experiment(v_size, FIXED_WIN, FIXED_HOP, epochs=15)
    vocab_results.append({
        'VOCAB_SIZE': v_size,
        'WIN_SIZE': FIXED_WIN,
        'WIN_HOP': FIXED_HOP,
        'Best Val Acc': round(acc, 4)
    })


df_vocab = pd.DataFrame(vocab_results)
print("\nИтоги эксперимента с VOCAB_SIZE:")
print(df_vocab)


--- Запуск: Vocab=5000, Win=1000, Hop=100 ---
Epoch 5/15 | Loss: 0.4619 | Val Acc: 0.3961
Epoch 10/15 | Loss: 0.0575 | Val Acc: 0.3751
Epoch 15/15 | Loss: 0.0055 | Val Acc: 0.3932

--- Запуск: Vocab=10000, Win=1000, Hop=100 ---
Epoch 5/15 | Loss: 0.2453 | Val Acc: 0.3660
Epoch 10/15 | Loss: 0.0180 | Val Acc: 0.3519
Epoch 15/15 | Loss: 0.0108 | Val Acc: 0.3515

--- Запуск: Vocab=20000, Win=1000, Hop=100 ---
Epoch 5/15 | Loss: 0.1424 | Val Acc: 0.4165
Epoch 10/15 | Loss: 0.0199 | Val Acc: 0.4337
Epoch 15/15 | Loss: 0.0096 | Val Acc: 0.4351

--- Запуск: Vocab=40000, Win=1000, Hop=100 ---
Epoch 5/15 | Loss: 0.0984 | Val Acc: 0.3536
Epoch 10/15 | Loss: 0.0071 | Val Acc: 0.3442
Epoch 15/15 | Loss: 0.0082 | Val Acc: 0.3358

Итоги эксперимента с VOCAB_SIZE:
   VOCAB_SIZE  WIN_SIZE  WIN_HOP  Best Val Acc
0        5000      1000      100        0.4370
1       10000      1000      100        0.4168
2       20000      1000      100        0.4352
3       40000      1000      100        0.3648


In [21]:
window_results = []
FIXED_VOCAB = 20000
WINDOW_PARAMS = [(500, 50), (1000, 100), (2000, 200)] 

for w_size, w_hop in WINDOW_PARAMS:
    acc = run_experiment(FIXED_VOCAB, w_size, w_hop, epochs=15)
    window_results.append({
        'VOCAB_SIZE': FIXED_VOCAB,
        'WIN_SIZE': w_size,
        'WIN_HOP': w_hop,
        'Best Val Acc': round(acc, 4)
    })


df_window = pd.DataFrame(window_results)
print("\nИтоги эксперимента с параметрами окна:")
print(df_window)


--- Запуск: Vocab=20000, Win=500, Hop=50 ---
Epoch 5/15 | Loss: 0.0579 | Val Acc: 0.3677
Epoch 10/15 | Loss: 0.0122 | Val Acc: 0.3164
Epoch 15/15 | Loss: 0.0253 | Val Acc: 0.3773

--- Запуск: Vocab=20000, Win=1000, Hop=100 ---
Epoch 5/15 | Loss: 0.1759 | Val Acc: 0.3947
Epoch 10/15 | Loss: 0.0122 | Val Acc: 0.4121
Epoch 15/15 | Loss: 0.0168 | Val Acc: 0.4264

--- Запуск: Vocab=20000, Win=2000, Hop=200 ---
Epoch 5/15 | Loss: 0.6365 | Val Acc: 0.2993
Epoch 10/15 | Loss: 0.0401 | Val Acc: 0.3754
Epoch 15/15 | Loss: 0.0082 | Val Acc: 0.3989

Итоги эксперимента с параметрами окна:
   VOCAB_SIZE  WIN_SIZE  WIN_HOP  Best Val Acc
0       20000       500       50        0.3962
1       20000      1000      100        0.4264
2       20000      2000      200        0.4617


In [22]:

final_df = pd.concat([df_vocab, df_window]).drop_duplicates().reset_index(drop=True)
print("Сводная таблица всех экспериментов:")
display(final_df)

Сводная таблица всех экспериментов:


,VOCAB_SIZE,WIN_SIZE,WIN_HOP,Best Val Acc
0,5000,1000,100,0.4370
1,10000,1000,100,0.4168
2,20000,1000,100,0.4352
3,40000,1000,100,0.3648
4,20000,500,50,0.3962
5,20000,1000,100,0.4264
6,20000,2000,200,0.4617
